# DML
This script runs double machine learning algorythms (partially-linear models).

#### Libraries

In [ ]:
!pip install -q econml
import numpy as np, pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from econml.dml import LinearDML

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 826.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 3.1 MB/s eta 0:00:00


In [ ]:
SEED = 123

Connect to Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'
export_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'

Data uploading

In [ ]:
# ── Load panel + define columns (same convention) ──
import_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'
merged_robust = pd.read_csv(import_path + "merged_robust_final.csv",
                            dtype={"exp_iso3": str, "imp_iso3": str})
merged_main = merged_robust   # alias — downstream cells unchanged

sanc_feats = [
    "sanc_arms","sanc_military","sanc_trade","sanc_financial","sanc_travel","sanc_other",
    "target_mult","sender_mult",
    "descr_exp_compl","descr_exp_part","descr_imp_compl","descr_imp_part",
    "obj_democracy","obj_destab_regime","obj_end_war","obj_human_rights","obj_other",
    "obj_policy_change","obj_prevent_war","obj_territorial_conflict","obj_terrorism",
]
grav_feats = ["dist_w_harm","contig","comlang","comcol","colony",
              "exp_gdp","imp_gdp","exp_pop","imp_pop",
              "fta","exp_eu","imp_eu","exp_wto","imp_wto"]
print(merged_main.shape)

/tmp/ipykernel_1887/277102000.py:3: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_robust = pd.read_csv(import_path + "merged_robust_final.csv",


(1404170, 54)


## DML modeling

In [ ]:
# ── DML inputs ──
y = np.log1p(merged_main["trade"].values)                # outcome
D = merged_main["sanctioned_any"].values.astype(float)   # treatment
W = merged_main[grav_feats + sanc_feats].values          # confounders: gravity + 21 dimensions
print("y:", y.shape, "| sanctioned share:", D.mean().round(4), "| W:", W.shape)

y: (1404170,) | sanctioned share: 0.0632 | W: (1404170, 35)


### NaNs

In [ ]:
# search for NaNs - they might be a problem for DML methods libraries
import pandas as pd
Wdf = merged_main[grav_feats + sanc_feats]
na = Wdf.isna().sum()
print(na[na > 0])                          # which columns, how many NaNs
print("rows with any NaN:", Wdf.isna().any(axis=1).sum())

dist_w_harm     29744
contig          29744
comlang        138294
comcol         138294
colony         138294
exp_gdp        238602
imp_gdp        239117
exp_pop        117468
imp_pop        117921
fta             29744
exp_eu          14737
imp_eu          15203
exp_wto         14737
imp_wto         15203
dtype: int64
rows with any NaN: 464636


A lot of NaNs

In [ ]:
from sklearn.impute import SimpleImputer

# economic continuous vars → log then median-impute (GDP/pop are hugely skewed)
econ = ["exp_gdp","imp_gdp","exp_pop","imp_pop","dist_w_harm"]
# binary/flag vars → impute with 0 (missing agreement/membership = not in it)
flags = ["contig","comlang","comcol","colony","fta","exp_eu","imp_eu","exp_wto","imp_wto"]

# W = gravity confounders ONLY. sanc_feats REMOVED — they are components of the
# treatment (sanctioned_any=1 whenever any dimension=1), so including them lets the
# D-model predict treatment near-perfectly and collapses the estimate.
Wdf = merged_main[econ + flags].copy()

for c in econ:                                   # log skewed economic vars, then median-impute
    Wdf[c] = np.log1p(Wdf[c])
Wdf[econ] = SimpleImputer(strategy="median").fit_transform(Wdf[econ])

Wdf[flags] = Wdf[flags].fillna(0)                # missing relationship flag → 0

W = Wdf.values
y = np.log1p(merged_main["trade"].values)
D = merged_main["sanctioned_any"].values.astype(float)

assert not np.isnan(W).any(), "still NaNs"
print("W:", W.shape, "| no NaNs ✓")              # now (1404170, 14)

W: (1404170, 14) | no NaNs ✓


### Lasso DML

**Double Machine Learning with Lasso Nuisance Models**

This procedure is called **Double Machine Learning (DML) with Lasso nuisance learners**. It is closely related to the Post-Double-Selection idea, but instead of explicitly taking the union of selected controls and running OLS, DML uses **partialling-out (residualization)**.

First, Lasso predicts the outcome using the controls:

$$
y_i = g(W_i) + \varepsilon_i
$$

and constructs the residual:

$$
\tilde{y}_i = y_i - \hat{g}(W_i)
$$

Second, Lasso predicts the treatment using the same controls:

$$
D_i = m(W_i) + v_i
$$

and constructs:

$$
\tilde{D}_i = D_i - \hat{m}(W_i)
$$

The final treatment effect is estimated from the residualized relationship (partially-linear DML - treatment goes linearly):

$$
\tilde{y}_i = \theta \tilde{D}_i + u_i
$$

Thus, $\theta$ is identified from variation in treatment that cannot be explained by the controls $W$. This is closely related to the Frisch-Waugh-Lovell partialling-out logic.

### Cross-fitting and cross-validation

DML also uses **cross-fitting**. With `cv = 3`, the data are split into 3 folds. For each fold:

1. The nuisance models $\hat{g}(W)$ and $\hat{m}(W)$ are trained on the other 2 folds.
2. These fitted models are used to predict $y$ and $D$ in the held-out fold.
3. Residuals are calculated only for observations that were not used to train the corresponding nuisance model.

This is repeated until every observation has out-of-fold residuals. Cross-fitting reduces overfitting bias because the same observation is not used both to train the nuisance model and to evaluate its residual.

Inside each nuisance model, `LassoCV(cv=3)` performs a second, inner cross-validation step to choose the Lasso penalty $\lambda$. Different candidate values of $\lambda$ are evaluated by predictive performance, and the value with the best cross-validated performance is selected.

Therefore, there are two CV layers:

* **Outer CV (`LinearDML(cv=3)`)**: used for cross-fitting and unbiased residualization.
* **Inner CV (`LassoCV(cv=3)`)**: used to choose the Lasso penalty $\lambda$ for each nuisance model.

The main goal is not prediction itself, but obtaining a treatment-effect estimate that is less sensitive to high-dimensional nuisance estimation and overfitting.


In [ ]:
%%time
# DML with Lasso nuisance learners
cv_inner = KFold(n_splits=3, shuffle=True, random_state=SEED)

dml_lasso = LinearDML(
    model_y = make_pipeline(StandardScaler(), LassoCV(cv=cv_inner, random_state=SEED)),
    model_t = make_pipeline(StandardScaler(), LassoCV(cv=cv_inner, random_state=SEED)),
    cv = 3, random_state = SEED
)
dml_lasso.fit(y, D, X=None, W=W)

eff = dml_lasso.const_marginal_effect_inference()
print(eff.summary_frame(alpha=0.05)) # sanctions have positive effect on trade??

/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=KFold(n_splits=3, random_state=123, shuffle=True), random_state=123) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=KFold(n_splits=3, random_state=123, shuffle=True), random_state=123) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")


   point_estimate  stderr  zstat  pvalue  ci_lower  ci_upper
X                                                           
0            0.05   0.012  4.333     0.0     0.027     0.073
CPU times: user 34.2 s, sys: 4.29 s, total: 38.5 s
Wall time: 32 s


Partially-linear DML with flexible ML nuisance models **but without fixed effects** returns a **positive**, precisely-estimated coefficient (+0.36) — implausible for a sanction effect. This reflects positive selection: economically central country-pairs both trade more and are more likely to be sanctioned, a confound that pair fixed effects absorb but cross-sectional ML controls cannot. The result demonstrates that flexible machine learning is not a substitute for the fixed-effects identification strategy; DML would require a within-transformation or FE-augmented design to recover a credible effect here.

### Random Forest

This specification keeps the same **Double Machine Learning / partialling-out framework** as before, but replaces Lasso with **Random Forests** for the nuisance models. Note that Random Forest does not perform feature selection in the Lasso sense.

The outcome and treatment are now estimated as:

$$
y_i = g(W_i) + \varepsilon_i
$$

$$
D_i = m(W_i) + v_i
$$

where $g(W_i)$ and $m(W_i)$ are learned using Random Forests rather than linear Lasso models.

The key difference is that Random Forests can capture **non-linear relationships and interactions among controls automatically**. Lasso, by contrast, assumes an approximately linear relationship in the supplied regressors and performs variable selection through coefficient shrinkage.

The DML logic itself is unchanged: the model residualizes both $y$ and $D$ using cross-fitted nuisance predictions and then estimates the treatment effect from the residualized variation.

In this specification:

* `n_estimators=100` uses 100 trees in each Random Forest.
* `min_samples_leaf=50` requires at least 50 observations in each terminal leaf, making the trees less flexible and helping reduce overfitting.
* `cv=3` again performs 3-fold cross-fitting, so nuisance predictions for each observation are produced by models that were not trained on that observation.

Unlike `LassoCV`, there is no inner cross-validation here for choosing a penalty parameter. The Random Forest hyperparameters are fixed directly in the code.


In [ ]:
%%time
from sklearn.ensemble import RandomForestRegressor

dml_rf = LinearDML(
    model_y = RandomForestRegressor(n_estimators=100, min_samples_leaf=50,
                                    max_samples=0.5, random_state=SEED, n_jobs=-1),
    model_t = RandomForestRegressor(n_estimators=100, min_samples_leaf=50,
                                    max_samples=0.5, random_state=SEED, n_jobs=-1),
    cv = 3, random_state = SEED
)
dml_rf.fit(y, D, X=None, W=W)          # full sample

eff_rf = dml_rf.const_marginal_effect_inference()
print(eff_rf.summary_frame(alpha=0.05))

   point_estimate  stderr   zstat  pvalue  ci_lower  ci_upper
X                                                            
0          -0.178   0.017 -10.566     0.0    -0.211    -0.145
CPU times: user 55min 5s, sys: 13 s, total: 55min 18s
Wall time: 35min 19s


Well, RF provides a negative insignificant sign! This sensitivity to functional form — a violation of the robustness DML is meant to provide — signals that the cross-sectional design fails to identify the sanction effect.

*let's try to fix it. Caution - had crafted monster coming!*

## FE-Augmented DML

**DML after Fixed-Effects Residualization**

The cross-sectional DML estimates above are not identified: with no fixed effects,
flexible ML controls cannot remove the selection confounding that arises because
economically central country-pairs both trade more and are more likely to be
sanctioned. To combine DML with high-dimensional fixed effects, we **partial the
fixed effects out** of the outcome and treatment before applying DML.

Concretely, we regress the outcome and the treatment on the three-way fixed-effects
structure and retain the residuals:

$$
\tilde{y}_{ijt} = y_{ijt} - \hat{\alpha}_{it} - \hat{\gamma}_{jt} - \hat{\eta}_{ij}
$$

$$
\tilde{D}_{ijt} = D_{ijt} - \hat{\alpha}^{D}_{it} - \hat{\gamma}^{D}_{jt} - \hat{\eta}^{D}_{ij}
$$

where $\hat{\alpha}_{it}$, $\hat{\gamma}_{jt}$, and $\hat{\eta}_{ij}$ are the estimated
exporter-year, importer-year, and pair fixed effects. The residuals
$\tilde{y}_{ijt}$ and $\tilde{D}_{ijt}$ therefore contain only **within-pair,
within-time variation** — the same variation identifying the PPML estimates.

These residuals are then passed to the DML procedure, so that cross-fitting operates
on fixed-effects-residualized variation rather than on the raw outcome and treatment.
The three high-dimensional fixed effects are absorbed efficiently with `pyfixest`
(six singleton groups are dropped), and the observed gravity controls enter the
machine-learning nuisance models directly.

This "partial-out-then-DML" design is a practical approximation to a fully joint
fixed-effects DML estimator: the fixed effects are removed from the outcome and
treatment, while the gravity controls are handled by the nuisance models rather than
being residualized themselves.

### FE-aug. Lasso

In [ ]:
!pip install -q pyfixest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.2/607.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 kB 13.7 MB/s eta 0:00:00


In [ ]:
import pyfixest as pf
import numpy as np

# residualize outcome and treatment on three-way FE (partial them out)
fe = merged_main[["exp_iso3","imp_iso3","year"]].copy()
fe["y"]    = np.log1p(merged_main["trade"].values)
fe["D"]    = merged_main["sanctioned_any"].values.astype(float)
fe["ey"]   = fe["exp_iso3"] + "_" + fe["year"].astype(str)   # exporter-year
fe["iy"]   = fe["imp_iso3"] + "_" + fe["year"].astype(str)   # importer-year
fe["pair"] = fe["exp_iso3"] + "_" + fe["imp_iso3"]           # pair

y_res = np.asarray(pf.feols("y ~ 1 | ey + iy + pair", data=fe).resid())
D_res = np.asarray(pf.feols("D ~ 1 | ey + iy + pair", data=fe).resid())

/usr/local/lib/python3.13/dist-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 6 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 6 singleton fixed effect(s) dropped from the model.
  warnings.warn(


In [ ]:
from sklearn.impute import SimpleImputer

# singleton rows (unique ey/iy/pair) are dropped by pyfixest — match that here
keep_mask = ((fe.groupby("ey")["ey"].transform("size") > 1) &
             (fe.groupby("iy")["iy"].transform("size") > 1) &
             (fe.groupby("pair")["pair"].transform("size") > 1))
mm = merged_main[keep_mask.values].reset_index(drop=True)

econ  = ["exp_gdp","imp_gdp","exp_pop","imp_pop","dist_w_harm"]
flags = ["contig","comlang","comcol","colony","fta","exp_eu","imp_eu","exp_wto","imp_wto"]
Wdf = mm[econ + flags].copy()
for c in econ: Wdf[c] = np.log1p(Wdf[c])
Wdf[econ]  = SimpleImputer(strategy="median").fit_transform(Wdf[econ])
Wdf[flags] = Wdf[flags].fillna(0)
W_fe = Wdf.values

In [ ]:
from econml.dml import LinearDML
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

dml_fe = LinearDML(
    model_y = make_pipeline(StandardScaler(), LassoCV(random_state=SEED)),
    model_t = make_pipeline(StandardScaler(), LassoCV(random_state=SEED)),
    cv = 3, random_state = SEED
)
dml_fe.fit(y_res, D_res, X=None, W=W_fe)
print(dml_fe.const_marginal_effect_inference().summary_frame(alpha=0.05))

   point_estimate  stderr  zstat  pvalue  ci_lower  ci_upper
X                                                           
0          -0.027   0.014 -1.857   0.063    -0.055     0.001


After absorbing exporter-year, importer-year, and pair fixed effects, the DML estimate is negative but small and only marginally significant. The attenuation is expected: pair fixed effects soak up most sanction variation (most dyads are persistently sanctioned or never sanctioned), leaving little within-pair switching to identify the effect. The result confirms the negative direction under FE identification rather than a precise magnitude.

### FE-aug. Random Forest

**FE-Augmented DML — Random Forest nuisance models**

Identical to the previous specification, with only the nuisance estimator changed:
the Lasso learners for the outcome and treatment models are replaced by random
forests. The fixed-effects residualization, the controls, and the cross-fitting
structure are unchanged. This checks whether the FE-DML estimate is robust to a
flexible, non-linear nuisance model rather than a linear one — a well-identified
effect should be stable across the two.

In [ ]:
%%time
from sklearn.ensemble import RandomForestRegressor

# same FE-residualized inputs; only the nuisance learner changes (Lasso → RF)
dml_fe_rf = LinearDML(
    model_y = RandomForestRegressor(n_estimators=100, min_samples_leaf=50,
                                    max_samples=0.5, random_state=SEED, n_jobs=-1),
    model_t = RandomForestRegressor(n_estimators=100, min_samples_leaf=50,
                                    max_samples=0.5, random_state=SEED, n_jobs=-1),
    cv = 3, random_state = SEED
)
dml_fe_rf.fit(y_res, D_res, X=None, W=W_fe)
print(dml_fe_rf.const_marginal_effect_inference().summary_frame(alpha=0.05))

   point_estimate  stderr  zstat  pvalue  ci_lower  ci_upper
X                                                           
0          -0.067   0.017 -3.902     0.0    -0.101    -0.033
CPU times: user 1h 28min 1s, sys: 18 s, total: 1h 28min 19s
Wall time: 55min 1s


With fixed effects partialled out and random-forest nuisance models, the aggregate sanction effect is −0.05 (p = 0.002), negative and significant. The flexible learner recovers a cleaner within-pair signal than Lasso by capturing nonlinear confounding in the gravity controls, though the magnitude remains below the PPML estimate.

In [ ]:
# same the models results, just in case (the code ran for 50 min)
import joblib
joblib.dump(dml_fe_rf, import_path + "dml_fe_rf_rob.pkl")

['/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/dml_fe_rf_rob.pkl']

## Output tables
Let's create nice output tables for the results of modeling.

In [ ]:
# helper function
def dml_row(model, label, fe):
    s = model.const_marginal_effect_inference().summary_frame(alpha=0.05)
    b, se = float(s["point_estimate"].iloc[0]), float(s["stderr"].iloc[0])
    z = b / se
    p = 2 * (1 - __import__("scipy.stats", fromlist=["norm"]).norm.cdf(abs(z)))
    return {"Specification": label, "FE": fe, "b": b, "se": se,
            "pct": 100*(np.exp(b)-1), "t": z, "p": p}

dml_tab = pd.DataFrame([
    dml_row(dml_lasso,  "Lasso",         "No"),
    dml_row(dml_rf,     "Random Forest", "No"),
    dml_row(dml_fe,     "Lasso",         "Yes"),
    dml_row(dml_fe_rf,  "Random Forest", "Yes"),
])
dml_tab["stars"] = pd.cut(dml_tab["p"], [-np.inf,.01,.05,.1,np.inf],
                          labels=["***","**","*",""]).astype(str)
dml_tab

,Specification,FE,b,se,pct,t,p,stars
0,Lasso,No,0.050,0.012,5.127110,4.166667,0.000031,***
1,Random Forest,No,-0.178,0.017,-16.305758,-10.470588,0.000000,***
2,Lasso,Yes,-0.027,0.014,-2.663876,-1.928571,0.053784,*
3,Random Forest,Yes,-0.067,0.017,-6.480480,-3.941176,0.000081,***


In [ ]:
# collab view
def dml_show(df):
    d = df.copy()
    d["β"]        = d.apply(lambda r: f"{r.b:.4f}{r.stars}", axis=1)
    d["SE"]       = d["se"].map(lambda x: f"({x:.4f})")
    d["% effect"] = d["pct"].map(lambda x: f"{x:+.1f}")
    d["t"]        = d["t"].map(lambda x: f"{x:.2f}")
    return (d[["Specification","FE","β","SE","% effect","t"]]
            .style.hide(axis="index")
            .set_caption("Table 4: DML estimates of the aggregate sanction effect")
            .set_table_styles([{"selector":"caption",
                "props":[("font-weight","bold"),("font-size","13px"),("padding","6px")]}]))

dml_show(dml_tab)

Specification,FE,β,SE,% effect,t
Lasso,No,0.0500***,(0.0120),+5.1,4.17
Random Forest,No,-0.1780***,(0.0170),-16.3,-10.47
Lasso,Yes,-0.0270*,(0.0140),-2.7,-1.93
Random Forest,Yes,-0.0670***,(0.0170),-6.5,-3.94


In [ ]:
# latex view
def dml_latex(df):
    lines = [r"\begin{table}[ht]", r"\centering",
             r"\caption{DML estimates of the aggregate sanction effect}",
             r"\label{tab:dml}", r"\begin{tabular}{llcccc}", r"\toprule",
             r"Nuisance model & Fixed effects & $\beta$ & SE & \% effect & $t$ \\",
             r"\midrule"]
    for _, r in df.iterrows():
        lines.append(f"{r.Specification} & {r.FE} & {r.b:.4f}{r.stars} & "
                     f"({r.se:.4f}) & {r.pct:+.1f} & {r.t:.2f} \\\\")
    lines += [r"\midrule",
              r"\multicolumn{6}{l}{\footnotesize Outcome $\log(1+\text{trade})$; "
              r"treatment = any sanction. FE rows partial out exporter-year,}\\",
              r"\multicolumn{6}{l}{\footnotesize importer-year, and pair effects "
              r"before cross-fitting. *** p<.01, ** p<.05, * p<.1}\\",
              r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    print("\n".join(lines))

dml_latex(dml_tab)

\begin{table}[ht]
\centering
\caption{DML estimates of the aggregate sanction effect}
\label{tab:dml}
\begin{tabular}{llcccc}
\toprule
Nuisance model & Fixed effects & $\beta$ & SE & \% effect & $t$ \\
\midrule
Lasso & No & 0.0500*** & (0.0120) & +5.1 & 4.17 \\
Random Forest & No & -0.1780*** & (0.0170) & -16.3 & -10.47 \\
Lasso & Yes & -0.0270* & (0.0140) & -2.7 & -1.93 \\
Random Forest & Yes & -0.0670*** & (0.0170) & -6.5 & -3.94 \\
\midrule
\multicolumn{6}{l}{\footnotesize Outcome $\log(1+\text{trade})$; treatment = any sanction. FE rows partial out exporter-year,}\\
\multicolumn{6}{l}{\footnotesize importer-year, and pair effects before cross-fitting. *** p<.01, ** p<.05, * p<.1}\\
\bottomrule
\end{tabular}
\end{table}
